# Science-path DM calibration and NCPA estimation from PSFs

The science camera sees the DM through other relay optics than the WFS, so the DM's rotation, flips, magnification, anamorphosis and shift on the camera need not match the WFS-path calibration. Static aberrations of the science path (non-common-path aberrations, NCPA) are invisible to the WFS. This notebook calibrates both from camera PSFs recorded for a set of known DM probe commands, with `TwinCalibrator`:

0. **Plate scale.** `estimate_psf_sampling` reads it (pixels per $\lambda/D$) from the OTF cutoff of a close-to-diffraction-limited frame.
1. **Rough orientation.** `rough_calibrate_dm_to_psfs` scans the rotation over $\pm 180°$ with and without a left-right flip, on low-order probes.
2. **Relay geometry.** `fit_dm_to_psfs` fits the geometry of a *copy* of the WFS-path DM jointly with the NCPA, a free tip-tilt per frame and a background. Sign, influence-function shape and coupling stay at the WFS-path values.
3. **NCPA.** `TwinCalibrator.estimate_ncpa` fits the NCPA (and optionally the pupil amplitude) with the DM fixed. The same method measures the NCPA on an already-calibrated system, without recalibrating anything.

The forward model is `WFS.GetPSF`. Design: `Ideas/21-psf-dm-misregistration.md` (supersedes the hand-written loop of `Ideas/06-ncpa-estimation.md`).

There is no real bench behind this notebook: the "camera" frames are synthesized with a known relay geometry, NCPA, per-frame tip-tilt, background and photon noise, so every fitted value can be compared with the truth. On a bench, replace the synthetic-data cell with the recorded frames (dark-subtracted, cropped to a square window roughly centred on the PSF, each normalized to sum 1) and the commands used to record them.

In [ ]:
import copy

import matplotlib.pyplot as plt
import torch
from mmengine import Config

from AI4AO import (DATA_DIR, WFS, DeformableMirror, TwinCalibrator, estimate_psf_sampling,
                   imshow_multiple)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
live_plot = True        # live fit plots; set False for a quiet run
torch.manual_seed(0)

## Configuration

`NCPAEstimation_params.py` holds the aperture (`WFSParams`; only `wfs.pupil`, `Nres`, `D` and `GetPSF` are used), the DM as calibrated on the WFS path (`DMParams`) and the probes and fit settings (`NCPAParams`). Its "demo only" keys only describe the synthetic data.

`dm_wfs` stands for the WFS-path calibrated DM. `pupil_amplitude_wfs` stands for the pupil amplitude that the WFS calibration estimated; on a bench, that is `wfs_calibrator.ref_pupil` from the WFS-path `TwinCalibrator`.

In [ ]:
paramfile = 'NCPAEstimation_params.py'
WFSParams = Config.fromfile(paramfile)['WFSParams']
DMParams = Config.fromfile(paramfile)['DMParams']
NCPAParams = Config.fromfile(paramfile)['NCPAParams']

wfs = WFS(WFSParams, device)
dm_wfs = DeformableMirror(WFSParams, DMParams, device)
dm_wfs.eval()

Nres = WFSParams["Nres"]
wavelength = NCPAParams["ScienceWavelength"]
nacts = int(dm_wfs.totalAct)
M2C = dm_wfs.MakeZernikeM2C(nModes=int(0.7 * nacts))        # (nacts, nModes), Noll ordering
ncpa_M2C = M2C[:, :NCPAParams["NCPAModes"]]                 # the NCPA basis: the first NCPAModes modes
pupil = wfs.pupil.bool()

pupil_amplitude_wfs = torch.ones(Nres, Nres, device=device)  # stand-in for the WFS-path estimate

## Probe commands

- One flat frame, used for the plate scale.
- $\pm$ pokes of low-order modes (focus to coma and beyond) for the rough search and the first geometry fit. Their capture range in rotation is large.
- $\pm$ pokes of higher-order modes, which pin the geometry down more precisely.
- A few random diversity commands, the `Ideas/06` style.

None of them pokes tip or tilt: each frame gets its own free tip-tilt, which would absorb such a poke entirely, so those frames would carry no information on the DM orientation.

`ProbeAmplitude` is a dimensionless `MakeZernikeM2C` coefficient, not meters: 1.0 is a ~1 rad poke.

In [ ]:
def pokes(columns, amplitude):
    return [sign * amplitude * M2C[:, col] for col in columns for sign in (1, -1)]

amplitude = NCPAParams["ProbeAmplitude"]
low_order = [torch.zeros(nacts, device=device)] + pokes(NCPAParams["LowOrderProbes"], amplitude)
high_order = pokes(NCPAParams["HighOrderProbes"], amplitude)
diversity = [M2C[:, 2:] @ (NCPAParams["DiversityAmplitude"] * torch.randn(M2C.shape[1] - 2, device=device))
             for _ in range(NCPAParams["NDiversity"])]

commands = torch.stack(low_order + high_order + diversity)   # (n_frames, nacts), zonal
low = slice(0, len(low_order))                                # frame 0 is the flat one
print(f"{len(commands)} frames ({len(low_order)} low-order), {nacts} actuators")

## Synthetic camera frames (demo only)

The truth that the calibration has to recover:

- **Relay geometry:** the science camera sees the DM flipped left-right, rotated by 37°, scaled by +2% / −1.5% (radial / tangential) and shifted.
- **NCPA:** a static DM-shaped aberration with content inside and outside the 10-mode basis, plus some tip-tilt. The tip-tilt is not NCPA: by design it ends up in the per-frame tip-tilt.
- **Pupil amplitude:** a 10% gradient that the WFS estimate doesn't have.
- **Per frame:** a random tip-tilt (centering and drift), a background and photon noise.

The plate scale is 130/48 ≈ 2.71 pixels per $\lambda/D$, a value `GetPSF` can produce exactly.

In [ ]:
dm_true = copy.deepcopy(dm_wfs)
dm_true.flip_lr = True
dm_true.rotationAngle = 37.0
dm_true.radialScaling = 0.02
dm_true.tangentialScaling = -0.015
dm_true.grid_shift = torch.tensor([[0.3, -0.2]], device=device)[..., None, None]

ncpa_coefs = torch.zeros(M2C.shape[1], device=device)
ncpa_coefs[0] = 0.3                                          # tilt: ends up in the per-frame tip-tilt
ncpa_coefs[2:10] = 0.15 * torch.randn(8, device=device)      # inside the NCPA basis
ncpa_coefs[10:20] = 0.03 * torch.randn(10, device=device)    # outside it
ncpa_command_true = M2C @ ncpa_coefs

amplitude_true = pupil_amplitude_wfs * (1 - 0.1 * torch.linspace(0, 1, Nres, device=device)[None, :])

c = torch.arange(Nres, device=device) - (Nres - 1) / 2
y, x = torch.meshgrid(c, c, indexing="ij")

def camera(commands, tiptilt):
    # Frames the camera would record: tiptilt in lambda/D (x = columns, y = rows).
    with torch.no_grad():
        tilt = wavelength / Nres * (tiptilt[:, 0, None, None] * x + tiptilt[:, 1, None, None] * y)
        psf = wfs.GetPSF(dm_true(commands + ncpa_command_true) + tilt, pupil=amplitude_true,
                         sampling=NCPAParams["TrueSampling"], wl=wavelength, fov_pix=NCPAParams["PSFWindow"])
        psf = psf / psf.sum(dim=(-2, -1), keepdim=True)
        frames = torch.poisson(psf * NCPAParams["Photons"] + NCPAParams["BackgroundCounts"])
    return frames / frames.sum(dim=(-2, -1), keepdim=True)   # normalized to sum 1, as the fit expects

tiptilt_true = 0.5 * torch.randn(len(commands), 2, device=device)
psfs = camera(commands, tiptilt_true)

def without_tiptilt(opd):
    # opd minus its least-squares piston/tip/tilt over the pupil
    basis = torch.stack([torch.ones_like(x), x, y])
    return opd - torch.einsum('k,kwh->wh', opd[pupil] @ torch.linalg.pinv(basis[:, pupil]), basis)

def true_ncpa(command=None):
    # The NCPA the fit should find (after applying a DM `command`): the true static OPD
    # without its piston/tip/tilt.
    with torch.no_grad():
        total = ncpa_command_true if command is None else ncpa_command_true + command
        return without_tiptilt(dm_true.GetDMShape(total[None])[0])

def rms_nm(opd):
    v = opd[pupil]
    return float((v - v.mean()).pow(2).mean().sqrt()) * 1e9

imshow_multiple([{"tensor": psfs[:4] ** 0.5, "title": "Camera frames (sqrt)"}], max_channel_number=4)
plt.show()

## Step 0: plate scale from the OTF cutoff

The OTF of the aperture vanishes beyond $D/\lambda$, i.e. beyond 1/sampling cycles per pixel. `estimate_psf_sampling` fits the shape of that edge on the flat frame. `GetPSF` can only produce samplings `Npix / Nres`, so the fit uses the closest one and prints the mismatch. A plate-scale error is partly absorbed by the DM scaling, since speckle positions scale with plate scale / magnification.

In [ ]:
sampling = estimate_psf_sampling(psfs[0])
print(f"Estimated sampling {sampling:.4f} pixels per lambda/D (truth {NCPAParams['TrueSampling']:.4f})")

## Steps 1 and 2: relay geometry of the science-path DM

`psf_calibrator` works on a **copy** of the WFS-path DM, so the WFS-path twin stays untouched. The rough search scans the rotation over ±180° in 10° steps, with and without a left-right flip, on the low-order probes. It never searches the DM sign: a PSF can't tell (geometry, sign) from (geometry rotated by 180°, opposite sign), so the sign stays the WFS-path value.

`fit_dm_to_psfs` then fits the relay geometry jointly with the NCPA (in `ncpa_M2C`), the per-frame tip-tilt and the background. The pupil amplitude starts from the WFS estimate and stays frozen here. The fit runs first on the low-order probes, then on all frames.

In [ ]:
psf_dm = copy.deepcopy(dm_wfs)
psf_calibrator = TwinCalibrator(wfs, psf_dm, device)

psf_calibrator.rough_calibrate_dm_to_psfs(commands[low], psfs[low], sampling, wavelength)

n_iter = NCPAParams["NIterations"]
_, positions_before, _ = psf_calibrator.fit_dm_to_psfs(
    commands[low], psfs[low], sampling, wavelength, static_M2C=ncpa_M2C,
    ref_pupil=pupil_amplitude_wfs, n_iter=n_iter, live_plot=live_plot)
loss, _, positions_after = psf_calibrator.fit_dm_to_psfs(
    commands, psfs, sampling, wavelength, static_M2C=ncpa_M2C, n_iter=n_iter, live_plot=live_plot)

In [ ]:
fitted, true = psf_dm.GetMisreg()[0], dm_true.GetMisreg()[0]
print(f"{'':20s}{'fitted':>10s}{'true':>10s}")
for key in ("rotationAngle", "radialScaling", "tangentialScaling", "anamorphosisAngle", "shiftX", "shiftY"):
    print(f"{key:20s}{fitted[key]:10.3f}{true[key]:10.3f}")
print(f"{'flip_lr':20s}{str(psf_dm.flip_lr):>10s}{str(dm_true.flip_lr):>10s}")

psf_calibrator.plot_actuator_and_offsets(positions_before, positions_after)
plt.show()

## Step 3: NCPA with the calibrated DM

`estimate_ncpa` fits only the NCPA, the per-frame tip-tilt, the background and here also the pupil amplitude (`fit_amplitude=True`), with the DM fixed. It runs on the same calibrator, so it starts from the pupil amplitude and NCPA of the geometry fit, and leaves the DM exactly as it was. The result is in the calibrator:

- `ref_opd`: the NCPA in meters, without piston/tip/tilt.
- `static_correction_command()`: the DM command that cancels it.
- `psf_tiptilt`, `psf_background`, `ref_pupil`.

`static_opd_mode="dm+pixel"` would also add a per-pixel map for what the 10-mode basis can't represent.

In [ ]:
psf_calibrator.estimate_ncpa(commands, psfs, sampling, wavelength, static_M2C=ncpa_M2C,
                             fit_amplitude=True, n_iter=n_iter, live_plot=live_plot)

with torch.no_grad():
    ncpa_true, ncpa_fit = true_ncpa(), psf_calibrator.ref_opd.detach()
print(f"NCPA {rms_nm(ncpa_true):.1f} nm RMS, residual {rms_nm(ncpa_true - ncpa_fit):.1f} nm RMS "
      "(the part outside the 10-mode basis can't be found in 'dm' mode)")
imshow_multiple([
    {"tensor": ncpa_true * pupil, "title": "True NCPA [m]", "same_scale": True},
    {"tensor": ncpa_fit * pupil, "title": "Fitted NCPA [m]", "same_scale": True},
    {"tensor": (ncpa_true - ncpa_fit) * pupil, "title": "Difference [m]", "scale_reference": ncpa_true * pupil},
])
plt.show()

## NCPA on an already-calibrated system

Later on, the NCPA can be measured again without recalibrating anything: reload the science-path DM, make a calibrator for it and call `estimate_ncpa` on new frames. (Use a calibrator of its own, never the WFS-path one: `estimate_ncpa` overwrites the calibrator's static offsets.)

The frames below are recorded with the first correction applied, and with a new tip-tilt per frame. Commands are absolute, i.e. they include the applied correction. So `estimate_ncpa` measures the **total** NCPA again, and its `static_correction_command()` **replaces** the applied correction rather than adding to it. Iterating like this refines the correction: the corrected PSFs are closer to the diffraction limit, and the probes are closer to the regime where the fit is most accurate.

In [ ]:
dm_path = DATA_DIR / "NCPAEstimation" / "NCPAEstimationDM_PSF.pth"
psf_dm.SaveCalibration(str(dm_path))

dm_loaded = DeformableMirror(WFSParams, DMParams, device)
dm_loaded.LoadCalibration(str(dm_path))
dm_loaded.eval()
state_before = {k: v.clone() for k, v in dm_loaded.state_dict().items()}

correction = psf_calibrator.static_correction_command()
commands_corrected = commands + correction
psfs_corrected = camera(commands_corrected, 0.5 * torch.randn(len(commands), 2, device=device))

reloaded = TwinCalibrator(wfs, dm_loaded, device)
reloaded.estimate_ncpa(commands_corrected, psfs_corrected, sampling, wavelength, static_M2C=ncpa_M2C,
                       ref_pupil=psf_calibrator.ref_pupil, n_iter=n_iter, live_plot=live_plot)
assert all(torch.equal(v, state_before[k]) for k, v in dm_loaded.state_dict().items()), "the DM changed"

second_correction = reloaded.static_correction_command()      # replaces the first one
print(f"Total NCPA measured again: {rms_nm(reloaded.ref_opd.detach()):.1f} nm RMS (truth {rms_nm(true_ncpa()):.1f} nm)")
print(f"NCPA left: {rms_nm(true_ncpa(correction)):.1f} nm after the first correction, "
      f"{rms_nm(true_ncpa(second_correction)):.1f} nm after the second (the part outside the 10-mode basis stays)")

flat = torch.zeros(1, nacts, device=device)
before, after = camera(flat, torch.zeros(1, 2, device=device)), camera(flat + correction, torch.zeros(1, 2, device=device))
imshow_multiple([{"tensor": before[0] ** 0.5, "title": "Flat DM (sqrt)", "same_scale": True},
                 {"tensor": after[0] ** 0.5, "title": "NCPA corrected (sqrt)", "same_scale": True}])
plt.show()
print(f"Peak intensity: {float(before.max()):.4f} -> {float(after.max()):.4f}")

## What the relay calibration buys

The same NCPA estimate, with the WFS-path DM instead of the science-path one. Its geometry is wrong for this camera: the NCPA fit absorbs part of the misregistration and the correction degrades.

In [ ]:
wfs_dm_calibrator = TwinCalibrator(wfs, dm_wfs, device)
wfs_dm_calibrator.estimate_ncpa(commands, psfs, sampling, wavelength, static_M2C=ncpa_M2C,
                                ref_pupil=pupil_amplitude_wfs, n_iter=n_iter, live_plot=False)
for name, result in (("science-path DM", psf_calibrator), ("WFS-path DM", wfs_dm_calibrator)):
    left = true_ncpa(result.static_correction_command())
    print(f"{name:16s}: NCPA left after its correction {rms_nm(left):6.1f} nm RMS")